# BeyondSmile: A Challenge on Detecting Depression through Facial Behavior and Head Gestures



In [109]:
import pandas as pd
import numpy as np
import tsfel
import neurokit2 as nk
import matplotlib.pyplot as plt
import datetime
import json
import pycatch22






# Read the columns for the data

In [110]:
columns_tself = pd.read_csv('./columns_mid_tsfel_euler.csv')

In [111]:
del columns_tself['Unnamed: 0']

In [112]:
tself_columns_mid = list(columns_tself.columns)


In [113]:
columns_pycatch = pd.read_csv('./columns_mid_pycatch_euler.csv')

In [114]:
del columns_pycatch['Unnamed: 0']

In [115]:
pycatch_columns_mid = list(columns_pycatch.columns)


In [116]:
tself_columns_mor = [name.replace('mid', 'mor') for name in tself_columns_mid]
tself_columns_aft = [name.replace('mid', 'aft') for name in tself_columns_mid]
tself_columns_eve = [name.replace('mid', 'eve') for name in tself_columns_mid]

In [117]:
pycatch_columns_mor = [name.replace('mid', 'mor') for name in pycatch_columns_mid]
pycatch_columns_aft = [name.replace('mid', 'aft') for name in pycatch_columns_mid]
pycatch_columns_eve = [name.replace('mid', 'eve') for name in pycatch_columns_mid]

# Read the labels

In [118]:
record = 7
data_phq = pd.read_csv('./dataset/groundtruth/phq9 _date.csv')
data_patient_depression = data_phq.loc[record]
patient = data_phq.loc[record]['pid']
diagnosis = data_phq.loc[record]['depression_episode']
start_monitoring = data_patient_depression.start_ts
end_monitoring = data_patient_depression.end_ts
#Change the dates into the timestamp
element_start = datetime.datetime(2022, int(start_monitoring.split('/')[0]), int(start_monitoring.split('/')[1]))
timestamp_start = datetime.datetime.timestamp(element_start)
element_end = datetime.datetime(2022, int(end_monitoring.split('/')[0]), int(end_monitoring.split('/')[1]))
timestamp_end = datetime.datetime.timestamp(element_end)
#Reorder the data
with open('./dataset/data/'+patient+ '.json', 'r') as f:
        data = json.load(f)

In [119]:
data_phq

,pid,start_ts,end_ts,start_phq9,end_phq9,depression_episode
0,P08,7/21/22,08/09/2022,6,1.0,0
1,P08,08/09/2022,8/23/22,1,9.0,0
2,P10,7/21/22,08/09/2022,8,7.0,1
3,P10,08/09/2022,09/02/2022,7,2.0,0
4,P12,7/22/22,08/09/2022,10,12.0,1
5,P12,08/09/2022,8/23/22,12,9.0,1
6,P13,7/25/22,08/09/2022,1,3.0,0
7,P13,08/09/2022,8/23/22,3,2.0,0
8,P14,7/25/22,08/08/2022,11,NaN,0
9,P15,7/26/22,08/10/2022,4,9.0,0


In [120]:
times = []
numbers = []
for i in range(0, len(data)):
    times.append(int(data[i]['timestamp'])/1000)
    numbers.append(i)
min_value = datetime.datetime.fromtimestamp(min(times)).isoformat()
max_value = datetime.datetime.fromtimestamp(max(times)).isoformat()

In [121]:
b = enumerate(times)
c = sorted(b, key = lambda i:i[1])
times_index_primary = []

for e in c:
    times_index_primary.append(e[0])

sorted_times = sorted(times)


# Reorder data json

In [122]:
data2 = []
for i in range(0, len(times_index_primary)):
    data2.append(data[times_index_primary[i]])

In [123]:
times = []
numbers = []
for i in range(0, len(data2)):
    times.append(int(data2[i]['timestamp'])/1000)
    numbers.append(i)

# Find the beginning of the record and end of the record

In [124]:
counter_start = 0
while(sorted_times[counter_start]<timestamp_start):
    counter_start +=1

In [125]:
counter_end = counter_start
for i in range(counter_start, len(sorted_times)):
    if sorted_times[counter_end]<=timestamp_end:
        counter_end +=1
    else:
        break
counter_end = counter_end - 1

In [126]:
#Select subdataset
data3 = data2[counter_start:counter_end+1]

In [127]:
timeline = []
timelinedate = []
for time in range(0, len(data3)):
    timeline.append(float(data3[time]['timestamp'])/1000)
    timelinedate.append(datetime.datetime.fromtimestamp(float(data3[time]['timestamp'])/1000))

In [128]:
len(data3)

429

In [129]:
def closest_index(target_value, timeline_list):
    differences = np.abs(np.array(timeline_list) - target_value)
    closest_index = differences.argmin()

    return closest_index
    

In [130]:
start_index_list = []
end_index_list = []

In [131]:
print(datetime.datetime.fromtimestamp(timeline[0]))
print(datetime.datetime.fromtimestamp(timeline[-1]))


2022-08-09 00:47:51.264000
2022-08-09 18:43:04.506000


In [132]:
start_index = 0

for portion in range(0, 100):

    flag =0
    start_value = timeline[start_index]


    target_value = start_value +60*60*24 #1 day more

    end_index = closest_index(target_value, timeline) 
    end_value = timeline[end_index]




    if (end_value - start_value) > 60*60*24:
        flag=1
        start_index_list.append(start_index)
        end_index_list.append(end_index - 1)
            
      #  print(datetime.datetime.fromtimestamp(timeline[start_index]))
      #  print(datetime.datetime.fromtimestamp(timeline[end_index-1]))
        
        start_index = end_index
    else:
        
        if (end_index+1)<len(timeline):
            if (timeline[end_index+1] - start_value) > 60*60*24:
                flag =1
                start_index_list.append(start_index)
                end_index_list.append(end_index)
                
             #   print(datetime.datetime.fromtimestamp(timeline[start_index]))
             #   print(datetime.datetime.fromtimestamp(timeline[end_index-1]))
            

                start_index = end_index +1

    
    

    if flag ==0:
        break
    
    
    print(portion)








In [133]:
sub_data = pd.DataFrame(columns=['record', 'start_subrecord', 'end_subrecord'])

In [134]:
sub_data['start_subrecord'] = start_index_list
sub_data['end_subrecord'] = end_index_list
sub_data['record'] = 0
sub_data['diagnosis'] = diagnosis

In [135]:
sub_data

,record,start_subrecord,end_subrecord,diagnosis


In [136]:
if len(sub_data)>0:
    sample = 0
else:
    sample = -1


In [137]:
if sample==0:
    start_index = sub_data.loc[sample]['start_subrecord']
    end_index = sub_data.loc[sample]['end_subrecord']
    data_test = data3[start_index:end_index+1]
    # Select the day for 4 periods: midnight (12am-6am), morning (6am-12pm), afternoon (12pm-6pm), and evening (6pm12am) (to daytime!)
    midnight_time = []
    morning_time = []
    afternoon_time = []
    evening_time = []
    for i in range(0, len(data_test)):
        hour_sample = datetime.datetime.fromtimestamp(float(data_test[i]['timestamp'])/1000).hour
        if hour_sample>=0 and hour_sample<6:
            midnight_time.append(i)
        if hour_sample>=6 and hour_sample<12:
            morning_time.append(i)
        if hour_sample>=12 and hour_sample<18:
            afternoon_time.append(i)
        if hour_sample>=18 and hour_sample<=23:
            evening_time.append(i)
        data_midnight = []
    for i in range(0, len(midnight_time)):
        data_midnight.append(data_test[midnight_time[i]])

    data_midnight = []
    if len(midnight_time)>0:
        for i in range(0, len(midnight_time)):
            data_midnight.append(data_test[midnight_time[i]])
            
    data_morning = []
    if len(morning_time)>0:
        for i in range(0, len(morning_time)):
            data_morning.append(data_test[morning_time[i]])
            
    data_afternoon = []
    if len(afternoon_time)>0:
        for i in range(0, len(afternoon_time)):
            data_afternoon.append(data_test[afternoon_time[i]])
            
    data_evening = []
    if len(evening_time)>0:
        for i in range(0, len(evening_time)):
            data_evening.append(data_test[evening_time[i]])
    # Define separete subdata for the midningt, morning, afternoon and evening 
    # Extract midnight featues for smiling and open eyes probabilities
    COLUMN_NAMES = ['X', 'Y', 'Z']
    COLUMN_NAMES_mid = []
    for i in range(0, len(COLUMN_NAMES)):
        COLUMN_NAMES_mid.append(COLUMN_NAMES[i] +'_mid')
    records_euler_mid = pd.DataFrame(columns= COLUMN_NAMES_mid)

    for i in range(0, len(data_midnight)):
        euler_data = data_midnight[i]['headEulerAngle']
        euler_values = list(euler_data.values())

        if len(euler_data)!=0:
            records_euler_mid.loc[i] = euler_values
        else: 
            records_euler_mid.loc[i] = [np.nan]*3
    records_euler_mid_cleaned = records_euler_mid.copy()
    records_euler_mid_cleaned = records_euler_mid_cleaned.dropna()
    if len(records_euler_mid_cleaned) >=12:     # Retrieves a pre-defined feature configuration file to extract the temporal, statistical and spectral feature sets
        cfg = tsfel.get_features_by_domain()

        # Extract features
        X = tsfel.time_series_features_extractor(cfg, records_euler_mid_cleaned)
        X_mid_to_delete = []
        for name in X.columns:
            if 'Spectrogram mean coefficient_' in name:
                X_mid_to_delete.append(name)
        X = X.drop(X_mid_to_delete, axis=1)

    else:
        X = pd.DataFrame(columns=tself_columns_mid)
        X.loc[0] = [np.nan]*372
    data_euler_mid = X.copy()
    if len(records_euler_mid_cleaned)>3:    
        for j in range(0, len(COLUMN_NAMES_mid)):
            name_euler = COLUMN_NAMES_mid[j]
            features_pycatch = pycatch22.catch22_all(records_euler_mid_cleaned[name_euler])
            COLUMN = []
            for i in range(0, len(features_pycatch['names'])):
                COLUMN.append(name_euler+ '_' + features_pycatch['names'][i]) 
            features_euler_sub_mid = pd.DataFrame(columns=COLUMN)
            features_euler_sub_mid.loc[0] = features_pycatch['values']
            if j == 0:
                features_euler_mid = features_euler_sub_mid.copy()
            else:
                features_euler_mid = pd.concat([features_euler_mid, features_euler_sub_mid], axis=1)
    else:
        features_euler_mid = pd.DataFrame(columns=pycatch_columns_mid)
        features_euler_mid.loc[0] = [np.nan]*66 

    data_euler_mid = pd.concat([data_euler_mid, features_euler_mid], axis=1)

    approx_entropy_columns = [name + '_app_ent' for name in records_euler_mid_cleaned.columns]
    data_approx_entropy_mid = pd.DataFrame(columns=approx_entropy_columns)
    app_ent_euler_mid= []

    for i in range(0, len(approx_entropy_columns)): 
        try:
            approximate_entropy, parameters = nk.entropy_approximate(records_euler_mid_cleaned[records_euler_mid_cleaned.columns[i]])
             # Approximate entropy
        except:
            approximate_entropy = 0
        app_ent_euler_mid.append(approximate_entropy)
    data_approx_entropy_mid.loc[0] = app_ent_euler_mid
    data_euler_mid = pd.concat([data_euler_mid, data_approx_entropy_mid], axis=1)

    rsd_columns_mid = [name + '_rsd' for name in records_euler_mid_cleaned.columns]
    data_rsd_mid = pd.DataFrame(columns=rsd_columns_mid)
    rsd_euler_mid = []
    for i in range(0, len(rsd_columns_mid)): 
        rsd = 100*np.std(records_euler_mid_cleaned[records_euler_mid_cleaned.columns[i]])/(np.mean(records_euler_mid_cleaned[records_euler_mid_cleaned.columns[i]])+0.00000000000000000000001)
        rsd_euler_mid.append(rsd)
    data_rsd_mid.loc[0] = rsd_euler_mid

    data_euler_mid = pd.concat([data_euler_mid, data_rsd_mid], axis=1)
    # For morning

    COLUMN_NAMES = ['X', 'Y', 'Z']
    COLUMN_NAMES_mor = []
    for i in range(0, len(COLUMN_NAMES)):
        COLUMN_NAMES_mor.append(COLUMN_NAMES[i] +'_mor')
    records_euler_mor = pd.DataFrame(columns= COLUMN_NAMES_mor)

    for i in range(0, len(data_morning)):
        euler_data = data_morning[i]['headEulerAngle']
        euler_values = list(euler_data.values())

        if len(euler_data)!=0:
            records_euler_mor.loc[i] = euler_values
        else: 
            records_euler_mor.loc[i] = [np.nan]*3

    records_euler_mor_cleaned = records_euler_mor.copy()
    records_euler_mor_cleaned = records_euler_mor_cleaned.dropna()

    if len(records_euler_mor_cleaned) >=12:     # Retrieves a pre-defined feature configuration file to extract the temporal, statistical and spectral feature sets
        cfg = tsfel.get_features_by_domain()

        # Extract features
        X = tsfel.time_series_features_extractor(cfg, records_euler_mor_cleaned)
        X_mor_to_delete = []
        for name in X.columns:
            if 'Spectrogram mean coefficient_' in name:
                X_mor_to_delete.append(name)
        X = X.drop(X_mor_to_delete, axis=1)

    else:
        X = pd.DataFrame(columns=tself_columns_mor)
        X.loc[0] = [np.nan]*372

    data_euler_mor = X.copy()

    if len(records_euler_mor_cleaned)>3:    
        for j in range(0, len(COLUMN_NAMES_mor)):
            name_euler = COLUMN_NAMES_mor[j]
            features_pycatch = pycatch22.catch22_all(records_euler_mor_cleaned[name_euler])
            COLUMN = []
            for i in range(0, len(features_pycatch['names'])):
                COLUMN.append(name_euler+ '_' + features_pycatch['names'][i]) 
            features_euler_sub_mor = pd.DataFrame(columns=COLUMN)
            features_euler_sub_mor.loc[0] = features_pycatch['values']
            if j == 0:
                features_euler_mor = features_euler_sub_mor.copy()
            else:
                features_euler_mor = pd.concat([features_euler_mor, features_euler_sub_mor], axis=1)
    else:
        features_euler_mor = pd.DataFrame(columns=pycatch_columns_mor)
        features_euler_mor.loc[0] = [np.nan]*66 

    data_euler_mor = pd.concat([data_euler_mor, features_euler_mor], axis=1)

    approx_entropy_columns = [name + '_app_ent' for name in records_euler_mor_cleaned.columns]
    data_approx_entropy_mor = pd.DataFrame(columns=approx_entropy_columns)
    app_ent_euler_mor= []

    for i in range(0, len(approx_entropy_columns)): 
        try:
            approximate_entropy, parameters = nk.entropy_approximate(records_euler_mor_cleaned[records_euler_mor_cleaned.columns[i]])
             # Approximate entropy
        except:
            approximate_entropy = 0
        app_ent_euler_mor.append(approximate_entropy)
    data_approx_entropy_mor.loc[0] = app_ent_euler_mor

    data_euler_mor = pd.concat([data_euler_mor, data_approx_entropy_mor], axis=1)

    rsd_columns_mor= [name + '_rsd' for name in records_euler_mor_cleaned.columns]
    data_rsd_mor = pd.DataFrame(columns=rsd_columns_mor)
    rsd_euler_mor = []
    for i in range(0, len(rsd_columns_mor)): 
        rsd = 100*np.std(records_euler_mor_cleaned[records_euler_mor_cleaned.columns[i]])/(np.mean(records_euler_mor_cleaned[records_euler_mor_cleaned.columns[i]])+0.00000000000000000000001)
        rsd_euler_mor.append(rsd)
    data_rsd_mor.loc[0] = rsd_euler_mor

    data_euler_mor = pd.concat([data_euler_mor, data_rsd_mor], axis=1)
    # Afternoon data for smiling and eyes probabilities
    COLUMN_NAMES = ['X', 'Y', 'Z']
    COLUMN_NAMES_aft = []
    for i in range(0, len(COLUMN_NAMES)):
        COLUMN_NAMES_aft.append(COLUMN_NAMES[i] +'_aft')
    records_euler_aft = pd.DataFrame(columns= COLUMN_NAMES_aft)

    for i in range(0, len(data_afternoon)):
        euler_data = data_afternoon[i]['headEulerAngle']
        euler_values = list(euler_data.values())

        if len(euler_data)!=0:
            records_euler_aft.loc[i] = euler_values
        else: 
            records_euler_aft.loc[i] = [np.nan]*3
    records_euler_aft_cleaned = records_euler_aft.copy()
    records_euler_aft_cleaned = records_euler_aft_cleaned.dropna()
    if len(records_euler_aft_cleaned) >=12:     # Retrieves a pre-defined feature configuration file to extract the temporal, statistical and spectral feature sets
        cfg = tsfel.get_features_by_domain()

        # Extract features
        X = tsfel.time_series_features_extractor(cfg, records_euler_aft_cleaned)
        X_aft_to_delete = []
        for name in X.columns:
            if 'Spectrogram mean coefficient_' in name:
                X_aft_to_delete.append(name)
        X = X.drop(X_aft_to_delete, axis=1)

    else:
        X = pd.DataFrame(columns=tself_columns_aft)
        X.loc[0] = [np.nan]*372
    data_euler_aft = X.copy()
    if len(records_euler_aft_cleaned)>0:        
        for j in range(0, len(COLUMN_NAMES_aft)):
            name_euler = COLUMN_NAMES_aft[j]
            features_pycatch = pycatch22.catch22_all(records_euler_aft_cleaned[name_euler])
            COLUMN = []
            for i in range(0, len(features_pycatch['names'])):
                COLUMN.append(name_euler+ '_' + features_pycatch['names'][i]) 
            features_euler_sub_aft = pd.DataFrame(columns=COLUMN)
            features_euler_sub_aft.loc[0] = features_pycatch['values']
            if j == 0:
                features_euler_aft = features_euler_sub_aft.copy()
            else:
                features_euler_aft = pd.concat([features_euler_aft, features_euler_sub_aft], axis=1)
    else:
        features_euler_aft = pd.DataFrame(columns=pycatch_columns_aft)
        features_euler_aft.loc[0] = [np.nan]*66 

    data_euler_aft = pd.concat([data_euler_aft, features_euler_aft], axis=1)
    approx_entropy_columns = [name + '_app_ent' for name in records_euler_aft_cleaned.columns]
    data_approx_entropy_aft = pd.DataFrame(columns=approx_entropy_columns)
    app_ent_euler_aft= []

    for i in range(0, len(approx_entropy_columns)): 
        try:
            approximate_entropy, parameters = nk.entropy_approximate(records_euler_aft_cleaned[records_euler_aft_cleaned.columns[i]])
             # Approximate entropy
        except:
            approximate_entropy = 0
        app_ent_euler_aft.append(approximate_entropy)
    data_approx_entropy_aft.loc[0] = app_ent_euler_aft

    data_euler_aft = pd.concat([data_euler_aft, data_approx_entropy_aft], axis=1)

    rsd_columns_aft= [name + '_aft' for name in records_euler_aft_cleaned.columns]
    data_rsd_aft = pd.DataFrame(columns=rsd_columns_aft)
    rsd_euler_aft = []
    for i in range(0, len(rsd_columns_aft)): 
        rsd = 100*np.std(records_euler_aft_cleaned[records_euler_aft_cleaned.columns[i]])/(np.mean(records_euler_aft_cleaned[records_euler_aft_cleaned.columns[i]])+0.00000000000000000000001)
        rsd_euler_aft.append(rsd)
    data_rsd_aft.loc[0] = rsd_euler_aft

    data_euler_aft = pd.concat([data_euler_aft, data_rsd_aft], axis=1)
    # Probabilities for evening
    COLUMN_NAMES = ['X', 'Y', 'Z']
    COLUMN_NAMES_eve = []
    for i in range(0, len(COLUMN_NAMES)):
        COLUMN_NAMES_eve.append(COLUMN_NAMES[i] +'_eve')
    records_euler_eve = pd.DataFrame(columns= COLUMN_NAMES_eve)

    for i in range(0, len(data_evening)):
        euler_data = data_evening[i]['headEulerAngle']
        euler_values = list(euler_data.values())

        if len(euler_data)!=0:
            records_euler_eve.loc[i] = euler_values
        else: 
            records_euler_eve.loc[i] = [np.nan]*3
    records_euler_eve_cleaned = records_euler_eve.copy()
    records_euler_eve_cleaned = records_euler_eve_cleaned.dropna()
    if len(records_euler_eve_cleaned) >=12:     # Retrieves a pre-defined feature configuration file to extract the temporal, statistical and spectral feature sets
        cfg = tsfel.get_features_by_domain()

        # Extract features
        X = tsfel.time_series_features_extractor(cfg, records_euler_eve_cleaned)
        X_eve_to_delete = []
        for name in X.columns:
            if 'Spectrogram mean coefficient_' in name:
                X_eve_to_delete.append(name)
        X = X.drop(X_eve_to_delete, axis=1)

    else:
        X = pd.DataFrame(columns=tself_columns_eve)
        X.loc[0] = [np.nan]*372

    data_euler_eve = X.copy()

    if len(records_euler_eve_cleaned)>0:    
        for j in range(0, len(COLUMN_NAMES_eve)):
            name_euler = COLUMN_NAMES_eve[j]
            features_pycatch = pycatch22.catch22_all(records_euler_eve_cleaned[name_euler])
            COLUMN = []
            for i in range(0, len(features_pycatch['names'])):
                COLUMN.append(name_euler+ '_' + features_pycatch['names'][i]) 
            features_euler_sub_eve = pd.DataFrame(columns=COLUMN)
            features_euler_sub_eve.loc[0] = features_pycatch['values']
            if j == 0:
                features_euler_eve = features_euler_sub_eve.copy()
            else:
                features_euler_eve = pd.concat([features_euler_eve, features_euler_sub_eve], axis=1)
    else:
        features_euler_eve = pd.DataFrame(columns=pycatch_columns_eve)
        features_euler_eve.loc[0] = [np.nan]*66 

    data_euler_eve = pd.concat([data_euler_eve, features_euler_eve], axis=1)   
    approx_entropy_columns = [name + '_app_ent' for name in records_euler_eve_cleaned.columns]
    data_approx_entropy_eve = pd.DataFrame(columns=approx_entropy_columns)
    app_ent_euler_eve= []

    for i in range(0, len(approx_entropy_columns)): 
        try:
            approximate_entropy, parameters = nk.entropy_approximate(records_euler_eve_cleaned[records_euler_eve_cleaned.columns[i]])
             # Approximate entropy
        except:
            approximate_entropy = 0
        app_ent_euler_eve.append(approximate_entropy)
    data_approx_entropy_eve.loc[0] = app_ent_euler_eve
    data_approx_entropy_eve

    data_euler_eve = pd.concat([data_euler_eve, data_approx_entropy_eve], axis=1)

    rsd_columns_eve= [name + '_rsd' for name in records_euler_eve_cleaned.columns]
    data_rsd_eve = pd.DataFrame(columns=rsd_columns_eve)
    rsd_euler_eve = []
    for i in range(0, len(rsd_columns_eve)): 
        rsd = 100*np.std(records_euler_eve_cleaned[records_euler_eve_cleaned.columns[i]])/(np.mean(records_euler_eve_cleaned[records_euler_eve_cleaned.columns[i]])+0.00000000000000000000001)
        rsd_euler_eve.append(rsd)
    data_rsd_eve.loc[0] = rsd_euler_eve

    data_euler_eve = pd.concat([data_euler_eve, data_rsd_eve], axis=1)
    # Concat all probabilites
    data_euler = pd.concat([data_euler_mid, data_euler_mor, data_euler_aft, data_euler_eve], axis=1)
    # Prepare the name of the columns for the exceptions
    information_record = pd.DataFrame(columns = ['patient', 'record', 'diagnosis', 'type_data', 'subrecord'])
    information_record.loc[0] = [patient, record, diagnosis, 'test', 0]
    data_euler = pd.concat([information_record, data_euler], axis=1, join='inner')
    data_all_euler = data_euler.copy()

In [138]:
if len(sub_data)>1:
    flag_sub = 0
else:
    flag_sub = -1


In [139]:
if flag_sub!=-1:
    for sample in range(1, len(sub_data)):
        print(sample)
        start_index = sub_data.loc[sample]['start_subrecord']
        end_index = sub_data.loc[sample]['end_subrecord']
        data_test = data3[start_index:end_index+1]
        # Select the day for 4 periods: midnight (12am-6am), morning (6am-12pm), afternoon (12pm-6pm), and evening (6pm12am) (to daytime!)
        midnight_time = []
        morning_time = []
        afternoon_time = []
        evening_time = []
        for i in range(0, len(data_test)):
            hour_sample = datetime.datetime.fromtimestamp(float(data_test[i]['timestamp'])/1000).hour
            if hour_sample>=0 and hour_sample<6:
                midnight_time.append(i)
            if hour_sample>=6 and hour_sample<12:
                morning_time.append(i)
            if hour_sample>=12 and hour_sample<18:
                afternoon_time.append(i)
            if hour_sample>=18 and hour_sample<=23:
                evening_time.append(i)
                
                
         
        data_midnight = []
        
        if len(midnight_time)>0:
            for i in range(0, len(midnight_time)):
                data_midnight.append(data_test[midnight_time[i]])
        data_morning = []
        
        if len(morning_time)>0:
            for i in range(0, len(morning_time)):
                data_morning.append(data_test[morning_time[i]])
                
        data_afternoon = []
        
        if len(afternoon_time)>0:
            for i in range(0, len(afternoon_time)):
                data_afternoon.append(data_test[afternoon_time[i]])
                
        data_evening = []
        
        if len(evening_time)>0:

            for i in range(0, len(evening_time)):
                data_evening.append(data_test[evening_time[i]])
                
        print('Testing data')
        print(len(data_midnight))
        print(len(data_morning))
        print(len(data_afternoon))
        print(len(data_evening))
                
        # Define separete subdata for the midningt, morning, afternoon and evening 
        # Extract midnight featues for smiling and open eyes probabilities
        COLUMN_NAMES = ['X', 'Y', 'Z']
        COLUMN_NAMES_mid = []
        for i in range(0, len(COLUMN_NAMES)):
            COLUMN_NAMES_mid.append(COLUMN_NAMES[i] +'_mid')
        records_euler_mid = pd.DataFrame(columns= COLUMN_NAMES_mid)

        for i in range(0, len(data_midnight)):
            euler_data = data_midnight[i]['headEulerAngle']
            euler_values = list(euler_data.values())

            if len(euler_data)!=0:
                records_euler_mid.loc[i] = euler_values
            else: 
                records_euler_mid.loc[i] = [np.nan]*3
        records_euler_mid_cleaned = records_euler_mid.copy()
        records_euler_mid_cleaned = records_euler_mid_cleaned.dropna()
        if len(records_euler_mid_cleaned) >=12:     # Retrieves a pre-defined feature configuration file to extract the temporal, statistical and spectral feature sets
            cfg = tsfel.get_features_by_domain()

            # Extract features
            X = tsfel.time_series_features_extractor(cfg, records_euler_mid_cleaned)
            X_mid_to_delete = []
            for name in X.columns:
                if 'Spectrogram mean coefficient_' in name:
                    X_mid_to_delete.append(name)
            X = X.drop(X_mid_to_delete, axis=1)

        else:
            X = pd.DataFrame(columns=tself_columns_mid)
            X.loc[0] = [np.nan]*372
        data_euler_mid = X.copy()
        if len(records_euler_mid_cleaned)>3:    
            for j in range(0, len(COLUMN_NAMES_mid)):
                name_euler = COLUMN_NAMES_mid[j]
                features_pycatch = pycatch22.catch22_all(records_euler_mid_cleaned[name_euler])
                COLUMN = []
                for i in range(0, len(features_pycatch['names'])):
                    COLUMN.append(name_euler+ '_' + features_pycatch['names'][i]) 
                features_euler_sub_mid = pd.DataFrame(columns=COLUMN)
                features_euler_sub_mid.loc[0] = features_pycatch['values']
                if j == 0:
                    features_euler_mid = features_euler_sub_mid.copy()
                else:
                    features_euler_mid = pd.concat([features_euler_mid, features_euler_sub_mid], axis=1)
        else:
            features_euler_mid = pd.DataFrame(columns=pycatch_columns_mid)
            features_euler_mid.loc[0] = [np.nan]*66 

        data_euler_mid = pd.concat([data_euler_mid, features_euler_mid], axis=1)

        approx_entropy_columns = [name + '_app_ent' for name in records_euler_mid_cleaned.columns]
        data_approx_entropy_mid = pd.DataFrame(columns=approx_entropy_columns)
        app_ent_euler_mid= []

        for i in range(0, len(approx_entropy_columns)): 
            try:
                approximate_entropy, parameters = nk.entropy_approximate(records_euler_mid_cleaned[records_euler_mid_cleaned.columns[i]])
                 # Approximate entropy
            except:
                approximate_entropy = 0
            app_ent_euler_mid.append(approximate_entropy)
        data_approx_entropy_mid.loc[0] = app_ent_euler_mid
        data_euler_mid = pd.concat([data_euler_mid, data_approx_entropy_mid], axis=1)

        rsd_columns_mid = [name + '_rsd' for name in records_euler_mid_cleaned.columns]
        data_rsd_mid = pd.DataFrame(columns=rsd_columns_mid)
        rsd_euler_mid = []
        for i in range(0, len(rsd_columns_mid)): 
            rsd = 100*np.std(records_euler_mid_cleaned[records_euler_mid_cleaned.columns[i]])/(np.mean(records_euler_mid_cleaned[records_euler_mid_cleaned.columns[i]])+0.00000000000000000000001)
            rsd_euler_mid.append(rsd)
        data_rsd_mid.loc[0] = rsd_euler_mid

        data_euler_mid = pd.concat([data_euler_mid, data_rsd_mid], axis=1)
        # For morning

        COLUMN_NAMES = ['X', 'Y', 'Z']
        COLUMN_NAMES_mor = []
        for i in range(0, len(COLUMN_NAMES)):
            COLUMN_NAMES_mor.append(COLUMN_NAMES[i] +'_mor')
        records_euler_mor = pd.DataFrame(columns= COLUMN_NAMES_mor)

        for i in range(0, len(data_morning)):
            euler_data = data_morning[i]['headEulerAngle']
            euler_values = list(euler_data.values())

            if len(euler_data)!=0:
                records_euler_mor.loc[i] = euler_values
            else: 
                records_euler_mor.loc[i] = [np.nan]*3

        records_euler_mor_cleaned = records_euler_mor.copy()
        records_euler_mor_cleaned = records_euler_mor_cleaned.dropna()

        if len(records_euler_mor_cleaned) >=12:     # Retrieves a pre-defined feature configuration file to extract the temporal, statistical and spectral feature sets
            cfg = tsfel.get_features_by_domain()

            # Extract features
            X = tsfel.time_series_features_extractor(cfg, records_euler_mor_cleaned)
            X_mor_to_delete = []
            for name in X.columns:
                if 'Spectrogram mean coefficient_' in name:
                    X_mor_to_delete.append(name)
            X = X.drop(X_mor_to_delete, axis=1)

        else:
            X = pd.DataFrame(columns=tself_columns_mor)
            X.loc[0] = [np.nan]*372

        data_euler_mor = X.copy()

        if len(records_euler_mor_cleaned)>3:    
            for j in range(0, len(COLUMN_NAMES_mor)):
                name_euler = COLUMN_NAMES_mor[j]
                features_pycatch = pycatch22.catch22_all(records_euler_mor_cleaned[name_euler])
                COLUMN = []
                for i in range(0, len(features_pycatch['names'])):
                    COLUMN.append(name_euler+ '_' + features_pycatch['names'][i]) 
                features_euler_sub_mor = pd.DataFrame(columns=COLUMN)
                features_euler_sub_mor.loc[0] = features_pycatch['values']
                if j == 0:
                    features_euler_mor = features_euler_sub_mor.copy()
                else:
                    features_euler_mor = pd.concat([features_euler_mor, features_euler_sub_mor], axis=1)
        else:
            features_euler_mor = pd.DataFrame(columns=pycatch_columns_mor)
            features_euler_mor.loc[0] = [np.nan]*66 

        data_euler_mor = pd.concat([data_euler_mor, features_euler_mor], axis=1)

        approx_entropy_columns = [name + '_app_ent' for name in records_euler_mor_cleaned.columns]
        data_approx_entropy_mor = pd.DataFrame(columns=approx_entropy_columns)
        app_ent_euler_mor= []

        for i in range(0, len(approx_entropy_columns)): 
            try:
                approximate_entropy, parameters = nk.entropy_approximate(records_euler_mor_cleaned[records_euler_mor_cleaned.columns[i]])
                 # Approximate entropy
            except:
                approximate_entropy = 0
            app_ent_euler_mor.append(approximate_entropy)
        data_approx_entropy_mor.loc[0] = app_ent_euler_mor

        data_euler_mor = pd.concat([data_euler_mor, data_approx_entropy_mor], axis=1)

        rsd_columns_mor= [name + '_rsd' for name in records_euler_mor_cleaned.columns]
        data_rsd_mor = pd.DataFrame(columns=rsd_columns_mor)
        rsd_euler_mor = []
        for i in range(0, len(rsd_columns_mor)): 
            rsd = 100*np.std(records_euler_mor_cleaned[records_euler_mor_cleaned.columns[i]])/(np.mean(records_euler_mor_cleaned[records_euler_mor_cleaned.columns[i]])+0.00000000000000000000001)
            rsd_euler_mor.append(rsd)
        data_rsd_mor.loc[0] = rsd_euler_mor

        data_euler_mor = pd.concat([data_euler_mor, data_rsd_mor], axis=1)
        # Afternoon data for smiling and eyes probabilities
        COLUMN_NAMES = ['X', 'Y', 'Z']
        COLUMN_NAMES_aft = []
        for i in range(0, len(COLUMN_NAMES)):
            COLUMN_NAMES_aft.append(COLUMN_NAMES[i] +'_aft')
        records_euler_aft = pd.DataFrame(columns= COLUMN_NAMES_aft)

        for i in range(0, len(data_afternoon)):
            euler_data = data_afternoon[i]['headEulerAngle']
            euler_values = list(euler_data.values())

            if len(euler_data)!=0:
                records_euler_aft.loc[i] = euler_values
            else: 
                records_euler_aft.loc[i] = [np.nan]*3
        records_euler_aft_cleaned = records_euler_aft.copy()
        records_euler_aft_cleaned = records_euler_aft_cleaned.dropna()
        if len(records_euler_aft_cleaned) >=12:     # Retrieves a pre-defined feature configuration file to extract the temporal, statistical and spectral feature sets
            cfg = tsfel.get_features_by_domain()

            # Extract features
            X = tsfel.time_series_features_extractor(cfg, records_euler_aft_cleaned)
            X_aft_to_delete = []
            for name in X.columns:
                if 'Spectrogram mean coefficient_' in name:
                    X_aft_to_delete.append(name)
            X = X.drop(X_aft_to_delete, axis=1)

        else:
            X = pd.DataFrame(columns=tself_columns_aft)
            X.loc[0] = [np.nan]*372
        data_euler_aft = X.copy()
        if len(records_euler_aft_cleaned)>3:        
            for j in range(0, len(COLUMN_NAMES_aft)):
                name_euler = COLUMN_NAMES_aft[j]
                features_pycatch = pycatch22.catch22_all(records_euler_aft_cleaned[name_euler])
                COLUMN = []
                for i in range(0, len(features_pycatch['names'])):
                    COLUMN.append(name_euler+ '_' + features_pycatch['names'][i]) 
                features_euler_sub_aft = pd.DataFrame(columns=COLUMN)
                features_euler_sub_aft.loc[0] = features_pycatch['values']
                if j == 0:
                    features_euler_aft = features_euler_sub_aft.copy()
                else:
                    features_euler_aft = pd.concat([features_euler_aft, features_euler_sub_aft], axis=1)
        else:
            features_euler_aft = pd.DataFrame(columns=pycatch_columns_aft)
            features_euler_aft.loc[0] = [np.nan]*66 

        data_euler_aft = pd.concat([data_euler_aft, features_euler_aft], axis=1)
        approx_entropy_columns = [name + '_app_ent' for name in records_euler_aft_cleaned.columns]
        data_approx_entropy_aft = pd.DataFrame(columns=approx_entropy_columns)
        app_ent_euler_aft= []

        for i in range(0, len(approx_entropy_columns)): 
            try:
                approximate_entropy, parameters = nk.entropy_approximate(records_euler_aft_cleaned[records_euler_aft_cleaned.columns[i]])
                 # Approximate entropy
            except:
                approximate_entropy = 0
            app_ent_euler_aft.append(approximate_entropy)
        data_approx_entropy_aft.loc[0] = app_ent_euler_aft

        data_euler_aft = pd.concat([data_euler_aft, data_approx_entropy_aft], axis=1)

        rsd_columns_aft= [name + '_aft' for name in records_euler_aft_cleaned.columns]
        data_rsd_aft = pd.DataFrame(columns=rsd_columns_aft)
        rsd_euler_aft = []
        for i in range(0, len(rsd_columns_aft)): 
            rsd = 100*np.std(records_euler_aft_cleaned[records_euler_aft_cleaned.columns[i]])/(np.mean(records_euler_aft_cleaned[records_euler_aft_cleaned.columns[i]])+0.00000000000000000000001)
            rsd_euler_aft.append(rsd)
        data_rsd_aft.loc[0] = rsd_euler_aft

        data_euler_aft = pd.concat([data_euler_aft, data_rsd_aft], axis=1)
        # Probabilities for evening
        COLUMN_NAMES = ['X', 'Y', 'Z']
        COLUMN_NAMES_eve = []
        for i in range(0, len(COLUMN_NAMES)):
            COLUMN_NAMES_eve.append(COLUMN_NAMES[i] +'_eve')
        records_euler_eve = pd.DataFrame(columns= COLUMN_NAMES_eve)

        for i in range(0, len(data_evening)):
            euler_data = data_evening[i]['headEulerAngle']
            euler_values = list(euler_data.values())

            if len(euler_data)!=0:
                records_euler_eve.loc[i] = euler_values
            else: 
                records_euler_eve.loc[i] = [np.nan]*3
        records_euler_eve_cleaned = records_euler_eve.copy()
        records_euler_eve_cleaned = records_euler_eve_cleaned.dropna()
        if len(records_euler_eve_cleaned) >=12:     # Retrieves a pre-defined feature configuration file to extract the temporal, statistical and spectral feature sets
            cfg = tsfel.get_features_by_domain()

            # Extract features
            X = tsfel.time_series_features_extractor(cfg, records_euler_eve_cleaned)
            X_eve_to_delete = []
            for name in X.columns:
                if 'Spectrogram mean coefficient_' in name:
                    X_eve_to_delete.append(name)
            X = X.drop(X_eve_to_delete, axis=1)

        else:
            X = pd.DataFrame(columns=tself_columns_eve)
            X.loc[0] = [np.nan]*372

        data_euler_eve = X.copy()

        if len(records_euler_eve_cleaned)>3:    
            for j in range(0, len(COLUMN_NAMES_eve)):
                name_euler = COLUMN_NAMES_eve[j]
                features_pycatch = pycatch22.catch22_all(records_euler_eve_cleaned[name_euler])
                COLUMN = []
                for i in range(0, len(features_pycatch['names'])):
                    COLUMN.append(name_euler+ '_' + features_pycatch['names'][i]) 
                features_euler_sub_eve = pd.DataFrame(columns=COLUMN)
                features_euler_sub_eve.loc[0] = features_pycatch['values']
                if j == 0:
                    features_euler_eve = features_euler_sub_eve.copy()
                else:
                    features_euler_eve = pd.concat([features_euler_eve, features_euler_sub_eve], axis=1)
        else:
            features_euler_eve = pd.DataFrame(columns=pycatch_columns_eve)
            features_euler_eve.loc[0] = [np.nan]*66 

        data_euler_eve = pd.concat([data_euler_eve, features_euler_eve], axis=1)   
        approx_entropy_columns = [name + '_app_ent' for name in records_euler_eve_cleaned.columns]
        data_approx_entropy_eve = pd.DataFrame(columns=approx_entropy_columns)
        app_ent_euler_eve= []

        for i in range(0, len(approx_entropy_columns)): 
            try:
                approximate_entropy, parameters = nk.entropy_approximate(records_euler_eve_cleaned[records_euler_eve_cleaned.columns[i]])
                 # Approximate entropy
            except:
                approximate_entropy = 0
            app_ent_euler_eve.append(approximate_entropy)
        data_approx_entropy_eve.loc[0] = app_ent_euler_eve
        data_approx_entropy_eve

        data_euler_eve = pd.concat([data_euler_eve, data_approx_entropy_eve], axis=1)

        rsd_columns_eve= [name + '_rsd' for name in records_euler_eve_cleaned.columns]
        data_rsd_eve = pd.DataFrame(columns=rsd_columns_eve)
        rsd_euler_eve = []
        for i in range(0, len(rsd_columns_eve)): 
            rsd = 100*np.std(records_euler_eve_cleaned[records_euler_eve_cleaned.columns[i]])/(np.mean(records_euler_eve_cleaned[records_euler_eve_cleaned.columns[i]])+0.00000000000000000000001)
            rsd_euler_eve.append(rsd)
        data_rsd_eve.loc[0] = rsd_euler_eve

        data_euler_eve = pd.concat([data_euler_eve, data_rsd_eve], axis=1)
        # Concat all probabilites
        data_euler = pd.concat([data_euler_mid, data_euler_mor, data_euler_aft, data_euler_eve], axis=1)
        # Prepare the name of the columns for the exceptions
        information_record = pd.DataFrame(columns = ['patient', 'record', 'diagnosis', 'type_data', 'subrecord'])
        information_record.loc[0] = [patient, record, diagnosis, 'test', sample]
        data_euler = pd.concat([information_record, data_euler], axis=1, join='inner')


        data_all_euler = pd.concat([data_all_euler, data_euler])




        #if sample==0:
        start_index = sub_data.loc[sample]['start_subrecord']
        end_index = sub_data.loc[sample]['end_subrecord']
        data_train = data3[0:start_index]
        print(len(data_train))
        # Select the day for 4 periods: midnight (12am-6am), morning (6am-12pm), afternoon (12pm-6pm), and evening (6pm12am) (to daytime!)
        midnight_time = []
        morning_time = []
        afternoon_time = []
        evening_time = []
        for i in range(0, len(data_train)):
            hour_sample = datetime.datetime.fromtimestamp(float(data_train[i]['timestamp'])/1000).hour
            if hour_sample>=0 and hour_sample<6:
                midnight_time.append(i)
            if hour_sample>=6 and hour_sample<12:
                morning_time.append(i)
            if hour_sample>=12 and hour_sample<18:
                afternoon_time.append(i)
            if hour_sample>=18 and hour_sample<=23:
                evening_time.append(i)
                


        data_midnight = []
        if len(midnight_time)>0:
            for i in range(0, len(midnight_time)):
                data_midnight.append(data_train[midnight_time[i]])
                
        data_morning = []
        if len(morning_time)>0:
            for i in range(0, len(morning_time)):
                data_morning.append(data_train[morning_time[i]])
                
        data_afternoon = []
        if len(afternoon_time)>0:

            for i in range(0, len(afternoon_time)):
                data_afternoon.append(data_train[afternoon_time[i]])
        
        data_evening = []

        if len(evening_time)>0:
            for i in range(0, len(evening_time)):
                data_evening.append(data_train[evening_time[i]])
                
        print('Training data')
        print(len(data_midnight))
        print(len(data_morning))
        print(len(data_afternoon))
        print(len(data_evening))
        # Define separete subdata for the midningt, morning, afternoon and evening 
        # Extract midnight featues for smiling and open eyes probabilities
        COLUMN_NAMES = ['X', 'Y', 'Z']
        COLUMN_NAMES_mid = []
        for i in range(0, len(COLUMN_NAMES)):
            COLUMN_NAMES_mid.append(COLUMN_NAMES[i] +'_mid')
        records_euler_mid = pd.DataFrame(columns= COLUMN_NAMES_mid)

        for i in range(0, len(data_midnight)):
            euler_data = data_midnight[i]['headEulerAngle']
            euler_values = list(euler_data.values())

            if len(euler_data)!=0:
                records_euler_mid.loc[i] = euler_values
            else: 
                records_euler_mid.loc[i] = [np.nan]*3
        records_euler_mid_cleaned = records_euler_mid.copy()
        records_euler_mid_cleaned = records_euler_mid_cleaned.dropna()
        if len(records_euler_mid_cleaned) >=12:     # Retrieves a pre-defined feature configuration file to extract the temporal, statistical and spectral feature sets
            cfg = tsfel.get_features_by_domain()

            # Extract features
            X = tsfel.time_series_features_extractor(cfg, records_euler_mid_cleaned)
            X_mid_to_delete = []
            for name in X.columns:
                if 'Spectrogram mean coefficient_' in name:
                    X_mid_to_delete.append(name)
            X = X.drop(X_mid_to_delete, axis=1)

        else:
            X = pd.DataFrame(columns=tself_columns_mid)
            X.loc[0] = [np.nan]*372
        data_euler_mid = X.copy()
        if len(records_euler_mid_cleaned)>3:    
            for j in range(0, len(COLUMN_NAMES_mid)):
                name_euler = COLUMN_NAMES_mid[j]
                features_pycatch = pycatch22.catch22_all(records_euler_mid_cleaned[name_euler])
                COLUMN = []
                for i in range(0, len(features_pycatch['names'])):
                    COLUMN.append(name_euler+ '_' + features_pycatch['names'][i]) 
                features_euler_sub_mid = pd.DataFrame(columns=COLUMN)
                features_euler_sub_mid.loc[0] = features_pycatch['values']
                if j == 0:
                    features_euler_mid = features_euler_sub_mid.copy()
                else:
                    features_euler_mid = pd.concat([features_euler_mid, features_euler_sub_mid], axis=1)
        else:
            features_euler_mid = pd.DataFrame(columns=pycatch_columns_mid)
            features_euler_mid.loc[0] = [np.nan]*66 

        data_euler_mid = pd.concat([data_euler_mid, features_euler_mid], axis=1)

        approx_entropy_columns = [name + '_app_ent' for name in records_euler_mid_cleaned.columns]
        data_approx_entropy_mid = pd.DataFrame(columns=approx_entropy_columns)
        app_ent_euler_mid= []

        for i in range(0, len(approx_entropy_columns)): 
            try:
                approximate_entropy, parameters = nk.entropy_approximate(records_euler_mid_cleaned[records_euler_mid_cleaned.columns[i]])
                 # Approximate entropy
            except:
                approximate_entropy = 0
            app_ent_euler_mid.append(approximate_entropy)
        data_approx_entropy_mid.loc[0] = app_ent_euler_mid
        data_euler_mid = pd.concat([data_euler_mid, data_approx_entropy_mid], axis=1)

        rsd_columns_mid = [name + '_rsd' for name in records_euler_mid_cleaned.columns]
        data_rsd_mid = pd.DataFrame(columns=rsd_columns_mid)
        rsd_euler_mid = []
        for i in range(0, len(rsd_columns_mid)): 
            rsd = 100*np.std(records_euler_mid_cleaned[records_euler_mid_cleaned.columns[i]])/(np.mean(records_euler_mid_cleaned[records_euler_mid_cleaned.columns[i]])+0.00000000000000000000001)
            rsd_euler_mid.append(rsd)
        data_rsd_mid.loc[0] = rsd_euler_mid

        data_euler_mid = pd.concat([data_euler_mid, data_rsd_mid], axis=1)
        # For morning

        COLUMN_NAMES = ['X', 'Y', 'Z']
        COLUMN_NAMES_mor = []
        for i in range(0, len(COLUMN_NAMES)):
            COLUMN_NAMES_mor.append(COLUMN_NAMES[i] +'_mor')
        records_euler_mor = pd.DataFrame(columns= COLUMN_NAMES_mor)

        for i in range(0, len(data_morning)):
            euler_data = data_morning[i]['headEulerAngle']
            euler_values = list(euler_data.values())

            if len(euler_data)!=0:
                records_euler_mor.loc[i] = euler_values
            else: 
                records_euler_mor.loc[i] = [np.nan]*3

        records_euler_mor_cleaned = records_euler_mor.copy()
        records_euler_mor_cleaned = records_euler_mor_cleaned.dropna()

        if len(records_euler_mor_cleaned) >=12:     # Retrieves a pre-defined feature configuration file to extract the temporal, statistical and spectral feature sets
            cfg = tsfel.get_features_by_domain()

            # Extract features
            X = tsfel.time_series_features_extractor(cfg, records_euler_mor_cleaned)
            X_mor_to_delete = []
            for name in X.columns:
                if 'Spectrogram mean coefficient_' in name:
                    X_mor_to_delete.append(name)
            X = X.drop(X_mor_to_delete, axis=1)

        else:
            X = pd.DataFrame(columns=tself_columns_mor)
            X.loc[0] = [np.nan]*372

        data_euler_mor = X.copy()

        if len(records_euler_mor_cleaned)>3:    
            for j in range(0, len(COLUMN_NAMES_mor)):
                name_euler = COLUMN_NAMES_mor[j]
                features_pycatch = pycatch22.catch22_all(records_euler_mor_cleaned[name_euler])
                COLUMN = []
                for i in range(0, len(features_pycatch['names'])):
                    COLUMN.append(name_euler+ '_' + features_pycatch['names'][i]) 
                features_euler_sub_mor = pd.DataFrame(columns=COLUMN)
                features_euler_sub_mor.loc[0] = features_pycatch['values']
                if j == 0:
                    features_euler_mor = features_euler_sub_mor.copy()
                else:
                    features_euler_mor = pd.concat([features_euler_mor, features_euler_sub_mor], axis=1)
        else:
            features_euler_mor = pd.DataFrame(columns=pycatch_columns_mor)
            features_euler_mor.loc[0] = [np.nan]*66 

        data_euler_mor = pd.concat([data_euler_mor, features_euler_mor], axis=1)

        approx_entropy_columns = [name + '_app_ent' for name in records_euler_mor_cleaned.columns]
        data_approx_entropy_mor = pd.DataFrame(columns=approx_entropy_columns)
        app_ent_euler_mor= []

        for i in range(0, len(approx_entropy_columns)): 
            try:
                approximate_entropy, parameters = nk.entropy_approximate(records_euler_mor_cleaned[records_euler_mor_cleaned.columns[i]])
                 # Approximate entropy
            except:
                approximate_entropy = 0
            app_ent_euler_mor.append(approximate_entropy)
        data_approx_entropy_mor.loc[0] = app_ent_euler_mor

        data_euler_mor = pd.concat([data_euler_mor, data_approx_entropy_mor], axis=1)

        rsd_columns_mor= [name + '_rsd' for name in records_euler_mor_cleaned.columns]
        data_rsd_mor = pd.DataFrame(columns=rsd_columns_mor)
        rsd_euler_mor = []
        for i in range(0, len(rsd_columns_mor)): 
            rsd = 100*np.std(records_euler_mor_cleaned[records_euler_mor_cleaned.columns[i]])/(np.mean(records_euler_mor_cleaned[records_euler_mor_cleaned.columns[i]])+0.00000000000000000000001)
            rsd_euler_mor.append(rsd)
        data_rsd_mor.loc[0] = rsd_euler_mor

        data_euler_mor = pd.concat([data_euler_mor, data_rsd_mor], axis=1)
        # Afternoon data for smiling and eyes probabilities
        COLUMN_NAMES = ['X', 'Y', 'Z']
        COLUMN_NAMES_aft = []
        for i in range(0, len(COLUMN_NAMES)):
            COLUMN_NAMES_aft.append(COLUMN_NAMES[i] +'_aft')
        records_euler_aft = pd.DataFrame(columns= COLUMN_NAMES_aft)

        for i in range(0, len(data_afternoon)):
            euler_data = data_afternoon[i]['headEulerAngle']
            euler_values = list(euler_data.values())

            if len(euler_data)!=0:
                records_euler_aft.loc[i] = euler_values
            else: 
                records_euler_aft.loc[i] = [np.nan]*3
        records_euler_aft_cleaned = records_euler_aft.copy()
        records_euler_aft_cleaned = records_euler_aft_cleaned.dropna()
        if len(records_euler_aft_cleaned) >=12:     # Retrieves a pre-defined feature configuration file to extract the temporal, statistical and spectral feature sets
            cfg = tsfel.get_features_by_domain()

            # Extract features
            X = tsfel.time_series_features_extractor(cfg, records_euler_aft_cleaned)
            X_aft_to_delete = []
            for name in X.columns:
                if 'Spectrogram mean coefficient_' in name:
                    X_aft_to_delete.append(name)
            X = X.drop(X_aft_to_delete, axis=1)

        else:
            X = pd.DataFrame(columns=tself_columns_aft)
            X.loc[0] = [np.nan]*372
        data_euler_aft = X.copy()
        if len(records_euler_aft_cleaned)>3:        
            for j in range(0, len(COLUMN_NAMES_aft)):
                name_euler = COLUMN_NAMES_aft[j]
                features_pycatch = pycatch22.catch22_all(records_euler_aft_cleaned[name_euler])
                COLUMN = []
                for i in range(0, len(features_pycatch['names'])):
                    COLUMN.append(name_euler+ '_' + features_pycatch['names'][i]) 
                features_euler_sub_aft = pd.DataFrame(columns=COLUMN)
                features_euler_sub_aft.loc[0] = features_pycatch['values']
                if j == 0:
                    features_euler_aft = features_euler_sub_aft.copy()
                else:
                    features_euler_aft = pd.concat([features_euler_aft, features_euler_sub_aft], axis=1)
        else:
            features_euler_aft = pd.DataFrame(columns=pycatch_columns_aft)
            features_euler_aft.loc[0] = [np.nan]*66 

        data_euler_aft = pd.concat([data_euler_aft, features_euler_aft], axis=1)
        approx_entropy_columns = [name + '_app_ent' for name in records_euler_aft_cleaned.columns]
        data_approx_entropy_aft = pd.DataFrame(columns=approx_entropy_columns)
        app_ent_euler_aft= []

        for i in range(0, len(approx_entropy_columns)): 
            try:
                approximate_entropy, parameters = nk.entropy_approximate(records_euler_aft_cleaned[records_euler_aft_cleaned.columns[i]])
                 # Approximate entropy
            except:
                approximate_entropy = 0
            app_ent_euler_aft.append(approximate_entropy)
        data_approx_entropy_aft.loc[0] = app_ent_euler_aft

        data_euler_aft = pd.concat([data_euler_aft, data_approx_entropy_aft], axis=1)

        rsd_columns_aft= [name + '_aft' for name in records_euler_aft_cleaned.columns]
        data_rsd_aft = pd.DataFrame(columns=rsd_columns_aft)
        rsd_euler_aft = []
        for i in range(0, len(rsd_columns_aft)): 
            rsd = 100*np.std(records_euler_aft_cleaned[records_euler_aft_cleaned.columns[i]])/(np.mean(records_euler_aft_cleaned[records_euler_aft_cleaned.columns[i]])+0.00000000000000000000001)
            rsd_euler_aft.append(rsd)
        data_rsd_aft.loc[0] = rsd_euler_aft

        data_euler_aft = pd.concat([data_euler_aft, data_rsd_aft], axis=1)
        # Probabilities for evening
        COLUMN_NAMES = ['X', 'Y', 'Z']
        COLUMN_NAMES_eve = []
        for i in range(0, len(COLUMN_NAMES)):
            COLUMN_NAMES_eve.append(COLUMN_NAMES[i] +'_eve')
        records_euler_eve = pd.DataFrame(columns= COLUMN_NAMES_eve)

        for i in range(0, len(data_evening)):
            euler_data = data_evening[i]['headEulerAngle']
            euler_values = list(euler_data.values())

            if len(euler_data)!=0:
                records_euler_eve.loc[i] = euler_values
            else: 
                records_euler_eve.loc[i] = [np.nan]*3
        records_euler_eve_cleaned = records_euler_eve.copy()
        records_euler_eve_cleaned = records_euler_eve_cleaned.dropna()
        if len(records_euler_eve_cleaned) >=12:     # Retrieves a pre-defined feature configuration file to extract the temporal, statistical and spectral feature sets
            cfg = tsfel.get_features_by_domain()

            # Extract features
            X = tsfel.time_series_features_extractor(cfg, records_euler_eve_cleaned)
            X_eve_to_delete = []
            for name in X.columns:
                if 'Spectrogram mean coefficient_' in name:
                    X_eve_to_delete.append(name)
            X = X.drop(X_eve_to_delete, axis=1)

        else:
            X = pd.DataFrame(columns=tself_columns_eve)
            X.loc[0] = [np.nan]*372

        data_euler_eve = X.copy()

        if len(records_euler_eve_cleaned)>3:    
            for j in range(0, len(COLUMN_NAMES_eve)):
                name_euler = COLUMN_NAMES_eve[j]
                features_pycatch = pycatch22.catch22_all(records_euler_eve_cleaned[name_euler])
                COLUMN = []
                for i in range(0, len(features_pycatch['names'])):
                    COLUMN.append(name_euler+ '_' + features_pycatch['names'][i]) 
                features_euler_sub_eve = pd.DataFrame(columns=COLUMN)
                features_euler_sub_eve.loc[0] = features_pycatch['values']
                if j == 0:
                    features_euler_eve = features_euler_sub_eve.copy()
                else:
                    features_euler_eve = pd.concat([features_euler_eve, features_euler_sub_eve], axis=1)
        else:
            features_euler_eve = pd.DataFrame(columns=pycatch_columns_eve)
            features_euler_eve.loc[0] = [np.nan]*66 

        data_euler_eve = pd.concat([data_euler_eve, features_euler_eve], axis=1)   
        approx_entropy_columns = [name + '_app_ent' for name in records_euler_eve_cleaned.columns]
        data_approx_entropy_eve = pd.DataFrame(columns=approx_entropy_columns)
        app_ent_euler_eve= []

        for i in range(0, len(approx_entropy_columns)): 
            try:
                approximate_entropy, parameters = nk.entropy_approximate(records_euler_eve_cleaned[records_euler_eve_cleaned.columns[i]])
                 # Approximate entropy
            except:
                approximate_entropy = 0
            app_ent_euler_eve.append(approximate_entropy)
        data_approx_entropy_eve.loc[0] = app_ent_euler_eve
        data_approx_entropy_eve

        data_euler_eve = pd.concat([data_euler_eve, data_approx_entropy_eve], axis=1)

        rsd_columns_eve= [name + '_rsd' for name in records_euler_eve_cleaned.columns]
        data_rsd_eve = pd.DataFrame(columns=rsd_columns_eve)
        rsd_euler_eve = []
        for i in range(0, len(rsd_columns_eve)): 
            rsd = 100*np.std(records_euler_eve_cleaned[records_euler_eve_cleaned.columns[i]])/(np.mean(records_euler_eve_cleaned[records_euler_eve_cleaned.columns[i]])+0.00000000000000000000001)
            rsd_euler_eve.append(rsd) 
        data_rsd_eve.loc[0] = rsd_euler_eve

        data_euler_eve = pd.concat([data_euler_eve, data_rsd_eve], axis=1)
        # Concat all probabilites
        data_euler = pd.concat([data_euler_mid, data_euler_mor, data_euler_aft, data_euler_eve], axis=1)
        # Prepare the name of the columns for the exceptions
        information_record = pd.DataFrame(columns = ['patient', 'record', 'diagnosis', 'type_data', 'subrecord'])
        information_record.loc[0] = [patient, record, diagnosis, 'train', sample]
        data_euler = pd.concat([information_record, data_euler], axis=1, join='inner')
        data_all_euler = pd.concat([data_all_euler, data_euler])

In [140]:
data_all_euler.to_csv('dataset/euler_cross/eul_'+str(record)+'_.csv')